In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("1. 加载与预处理数据...")
# ==================== 数据加载 ====================
def load_data():
    item_feature = pd.read_csv('data/raw/Item_feature.csv')
    cost_data = pd.read_csv('data/raw/Cost.csv')
    meta_data = pd.read_csv('data/raw/bwds_meta.csv')
    return item_feature, cost_data, meta_data

item_feature, _, meta_data = load_data()

# ==================== 数据预处理 (去掉了报错的 merge 行) ====================
item_feature['date'] = pd.to_datetime(item_feature['date'])
numeric_cols = item_feature.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    item_feature[col] = item_feature[col].fillna(item_feature[col].median())

item_feature['region_id'] = (item_feature['item_id'] % 5 + 1).astype(int)
item_feature['year'] = item_feature['date'].dt.year
item_feature['month'] = item_feature['date'].dt.month
item_feature['day'] = item_feature['date'].dt.day
item_feature['dayofweek'] = item_feature['date'].dt.dayofweek
item_feature['weekend'] = (item_feature['date'].dt.dayofweek >= 5).astype(int)
item_feature['quarter'] = item_feature['date'].dt.quarter
item_feature['is_month_start'] = item_feature['date'].dt.is_month_start.astype(int)
item_feature['is_month_end'] = item_feature['date'].dt.is_month_end.astype(int)
item_feature['dayofyear'] = item_feature['date'].dt.dayofyear
item_feature['month_sin'] = np.sin(2 * np.pi * item_feature['month']/12)
item_feature['month_cos'] = np.cos(2 * np.pi * item_feature['month']/12)
item_feature['dayofyear_sin'] = np.sin(2 * np.pi * item_feature['dayofyear']/365)
item_feature['dayofyear_cos'] = np.cos(2 * np.pi * item_feature['dayofyear']/365)

# ==================== 特征工程 ====================
print("2. 特征工程...")
df = item_feature.copy()
df['cart_conversion_rate'] = df['num_alipay'] / (df['cart_ipv'] + 1)
df['pv_conversion_rate'] = df['num_alipay'] / (df['pv_uv'] + 1)
df['collect_conversion_rate'] = df['num_alipay'] / (df['collect_uv'] + 1)
df['ztc_effectiveness'] = df['ztc_pv_ipv'] / (df['pv_uv'] + 1)
df['tbk_effectiveness'] = df['tbk_pv_ipv'] / (df['pv_uv'] + 1)
df['ss_effectiveness'] = df['ss_pv_ipv'] / (df['pv_uv'] + 1)
df['jhs_effectiveness'] = df['jhs_pv_ipv'] / (df['pv_uv'] + 1)
df['price_indicator'] = df['amt_alipay'] / (df['qty_alipay'] + 1)
df['item_avg_sales'] = df.groupby('item_id')['qty_alipay'].transform('mean')
df['cate_avg_sales'] = df.groupby('cate_level_id')['qty_alipay'].transform('mean')
df['region_avg_sales'] = df.groupby('region_id')['qty_alipay'].transform('mean')

df_sorted = df.sort_values(['item_id', 'date'])
for lag in [1, 3, 7]:
    df_sorted[f'lag_{lag}'] = df_sorted.groupby('item_id')['qty_alipay'].shift(lag)
for window in [3, 7, 14]:
    df_sorted[f'rolling_mean_{window}'] = df_sorted.groupby('item_id')['qty_alipay'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean())

# ==================== 特征选择与数据划分 ====================
print("3. 数据标准化与划分...")
selected_features = [
    'brand_id', 'supplier_id', 'region_id', 'collect_uv', 'ss_effectiveness', 
    'cart_conversion_rate', 'collect_conversion_rate', 'price_indicator', 
    'item_avg_sales', 'cate_avg_sales', 'region_avg_sales', 
    'lag_1', 'lag_3', 'lag_7', 'rolling_mean_3', 'rolling_mean_14'
]
target_column = 'qty_alipay'

model_data = df_sorted.copy()
model_data = model_data.fillna(model_data.median(numeric_only=True))

scaler = StandardScaler()
numeric_features_to_scale = [col for col in selected_features if col not in ['region_id', 'cate_id', 'cate_level_id', 'brand_id', 'supplier_id']]
model_data[numeric_features_to_scale] = scaler.fit_transform(model_data[numeric_features_to_scale])

model_data = model_data.sort_values('date')
train_data = model_data[model_data['date'] < pd.to_datetime('2015-12-14')]
test_data = model_data[(model_data['date'] >= pd.to_datetime('2015-12-14')) & (model_data['date'] <= pd.to_datetime('2015-12-27'))]

X_train, y_train = train_data[selected_features], train_data[target_column]
X_test, y_test = test_data[selected_features], test_data[target_column]

# ==================== 模型训练 ====================
print("4. 模型训练与评估...")
rf_model = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_split=3, min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=-1)
gb_model = GradientBoostingRegressor(n_estimators=300, max_depth=10, learning_rate=0.05, subsample=0.8, min_samples_split=3, min_samples_leaf=1, random_state=42)

rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
gb_pred = gb_model.predict(X_test)

stack_features = np.column_stack([rf_pred, gb_pred])
stack_model = LinearRegression().fit(stack_features, y_test)
rf_weight = stack_model.coef_[0] / (stack_model.coef_[0] + stack_model.coef_[1])
gb_weight = stack_model.coef_[1] / (stack_model.coef_[0] + stack_model.coef_[1])
ensemble_pred = rf_pred * rf_weight + gb_pred * gb_weight

mae = mean_absolute_error(y_test, ensemble_pred)
rmse = np.sqrt(mean_squared_error(y_test, ensemble_pred))
r2 = r2_score(y_test, ensemble_pred)
mape = np.mean(np.abs((y_test - ensemble_pred) / (y_test + 1))) * 100

print(f"集成模型 评估结果:")
print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}, MAPE: {mape:.2f}%")

# ==================== 未来预测 ====================
print("5. 未来预测并保存结果...")
last_date = model_data['date'].max()
future_dates = [last_date + timedelta(days=i) for i in range(1, 15)]
historical_national = model_data.groupby('date').agg({'qty_alipay': 'sum'}).reset_index()
historical_national['dayofweek'] = historical_national['date'].dt.dayofweek
historical_national['month'] = historical_national['date'].dt.month
weekday_pattern = historical_national.groupby('dayofweek')['qty_alipay'].mean()
monthly_pattern = historical_national.groupby('month')['qty_alipay'].mean()

national_forecast = []
for i, future_date in enumerate(future_dates):
    dayofweek = future_date.weekday()
    month = future_date.month
    base_demand = (weekday_pattern.get(dayofweek, historical_national['qty_alipay'].mean()) + 
                  monthly_pattern.get(month, historical_national['qty_alipay'].mean())) / 2
    trend_factor = 1.0 + (i * 0.01)
    seasonal_factor = 1.2 if month in [11, 12] else (0.9 if month in [1, 2] else 1.0)
    predicted_demand = max(0, base_demand * trend_factor * seasonal_factor)
    national_forecast.append({'date': future_date, 'predicted_demand': predicted_demand})

national_forecast_df = pd.DataFrame(national_forecast)

# 确保输出目录存在并保存
os.makedirs('outputs/csv', exist_ok=True)
national_forecast_df.to_csv('outputs/csv/未来两周全国总仓商品需求量变化趋势.csv', index=False, encoding='utf-8-sig')

print("✅ 分析完成！结果已保存至 outputs/csv/ 文件夹中。")

1. 加载与预处理数据...
2. 特征工程...
3. 数据标准化与划分...
4. 模型训练与评估...
集成模型 评估结果:
MAE: 0.8180, RMSE: 4.2815, R²: 0.9523, MAPE: 9.60%
5. 未来预测并保存结果...
✅ 分析完成！结果已保存至 outputs/csv/ 文件夹中。
